In [1]:
import spacy
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from gensim import corpora
from gensim.models import LdaModel
from sklearn.feature_extraction.text import CountVectorizer

# Load spaCy model
nlp = spacy.load('en_core_web_sm')

# Initialize NLTK's SentimentIntensityAnalyzer
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/ThanhNguyen/nltk_data...


In [2]:
def extract_entities(text):
    doc = nlp(text)
    entities = {ent.label_: ent.text for ent in doc.ents}
    return entities

# Sample ESG text for NER
esg_text = """Our company has reduced carbon emissions by 20% compared to last year. 
We achieved a workforce diversity rate of 50%. 
Our renewable energy resources comprise 70% of our total energy usage."""

# Extract entities
entities = extract_entities(esg_text)
print("Extracted Entities:", entities)

Extracted Entities: {'PERCENT': '70%', 'DATE': 'last year'}


In [3]:
def topic_modeling(texts):
    # Tokenize the documents
    tokens = [nltk.word_tokenize(text.lower()) for text in texts]
    # Remove stopwords
    stop_words = nltk.corpus.stopwords.words('english')
    tokens = [[word for word in doc if word.isalnum() and word not in stop_words] for doc in tokens]

    # Create a dictionary and corpus for LDA
    dictionary = corpora.Dictionary(tokens)
    corpus = [dictionary.doc2bow(token) for token in tokens]

    # Train LDA model
    lda_model = LdaModel(corpus, num_topics=2, id2word=dictionary, passes=10)
    topics = lda_model.print_topics()
    return topics

# Sample ESG documents for topic modeling
esg_documents = [
    "Our company has reduced carbon emissions by 20%.",
    "We have a workforce diversity rate of 50%.",
    "Renewable energy resources comprise 70% of our energy usage."
]

# Extract topics
topics = topic_modeling(esg_documents)
print("Extracted Topics:")
for topic in topics:
    print(topic)

Extracted Topics:
(0, '0.135*"energy" + 0.081*"usage" + 0.081*"renewable" + 0.081*"comprise" + 0.081*"resources" + 0.081*"70" + 0.081*"workforce" + 0.081*"rate" + 0.081*"50" + 0.081*"diversity"')
(1, '0.119*"company" + 0.119*"reduced" + 0.119*"carbon" + 0.119*"emissions" + 0.119*"20" + 0.041*"diversity" + 0.041*"50" + 0.041*"rate" + 0.041*"workforce" + 0.040*"energy"')


In [4]:
def analyze_sentiment(text):
    sentiment_score = sia.polarity_scores(text)
    return sentiment_score

# Analyze sentiment of ESG text
sentiment = analyze_sentiment(esg_text)
print("Sentiment Analysis Results:", sentiment)

Sentiment Analysis Results: {'neg': 0.0, 'neu': 0.87, 'pos': 0.13, 'compound': 0.4939}


In [5]:
def run_esg_nlp_pipeline(esg_texts):
    # Extract entities
    entities = extract_entities(esg_texts)

    # Perform topic modeling
    topics = topic_modeling(esg_texts.split('. '))

    # Analyze sentiment
    sentiment = analyze_sentiment(esg_texts)

    return {
        'entities': entities,
        'topics': topics,
        'sentiment': sentiment
    }

# Run the complete pipeline
pipeline_results = run_esg_nlp_pipeline(esg_text)
print("NLP Pipeline Results:", pipeline_results)

NLP Pipeline Results: {'entities': {'PERCENT': '70%', 'DATE': 'last year'}, 'topics': [(0, '0.138*"energy" + 0.083*"usage" + 0.083*"resources" + 0.083*"comprise" + 0.083*"total" + 0.083*"renewable" + 0.083*"70" + 0.028*"50" + 0.028*"reduced" + 0.028*"achieved"'), (1, '0.065*"company" + 0.065*"20" + 0.065*"year" + 0.065*"compared" + 0.065*"emissions" + 0.065*"carbon" + 0.065*"rate" + 0.065*"last" + 0.065*"workforce" + 0.065*"diversity"')], 'sentiment': {'neg': 0.0, 'neu': 0.87, 'pos': 0.13, 'compound': 0.4939}}


In [1]:
import re
import json

# Define ESG indicators with their explanations
esg_indicators = {
    'carbon_emissions': "The total amount of carbon emissions in tons.",
    'diversity_ratio': "The percentage of diverse individuals in the workforce.",
    'renewable_energy_usage': "The percentage of energy used that comes from renewable sources.",
    'women_in_leadership': "The percentage of women in leadership positions."
}

# ESG text from which to extract values
esg_text = """
Our company emitted 2000 tons of CO2 in 2023, which reflects a significant reduction in carbon emissions.
The diversity ratio of our workforce is now 45%, showing improvement in inclusivity.
We have achieved 70% renewable energy usage this year, a notable increase from last year.
Currently, women occupy 35% of leadership roles in our company, surpassing prior goals.
"""

# Function to extract values based on ESG indicators
def extract_esg_values(indicators, text):
    extracted_values = {}
    for key, explanation in indicators.items():
        # Create a regex pattern for each key
        if key == 'carbon_emissions':
            pattern = r'(\d+)\s*tons? of CO2'
        elif key == 'diversity_ratio':
            pattern = r'(\d+)%\s*diversity ratio'
        elif key == 'renewable_energy_usage':
            pattern = r'(\d+)%\s*renewable energy usage'
        elif key == 'women_in_leadership':
            pattern = r'(\d+)%\s*of leadership roles'
        
        # Search for the pattern in the text
        match = re.search(pattern, text)
        if match:
            extracted_values[key] = match.group(1)

    return extracted_values

# Extract the values
esg_values = extract_esg_values(esg_indicators, esg_text)

# Convert to JSON format
esg_values_json = json.dumps(esg_values, indent=4)
print(esg_values_json)

{
    "carbon_emissions": "2000",
    "renewable_energy_usage": "70",
    "women_in_leadership": "35"
}
